In [2]:
import numpy as np
import pandas as pd

import os
import polars as pl
os.environ['POLARS_MAX_THREADS'] = '1'
print(pl.thread_pool_size())
from datetime import datetime, timedelta

1


In [4]:
DATA_DIR = "csv_files/"

trans = pd.read_csv(
    os.path.join(DATA_DIR, "mult_cohort_transaction_data.csv"), parse_dates = ['cohort_cutoff_date']
)



In [5]:
trans.info(show_counts = True)

<class 'pandas.DataFrame'>
RangeIndex: 16980367 entries, 0 to 16980366
Data columns (total 19 columns):
 #   Column                   Non-Null Count     Dtype         
---  ------                   --------------     -----         
 0   msno                     16980367 non-null  str           
 1   is_churn                 16980367 non-null  int64         
 2   num_transactions         16980367 non-null  int64         
 3   total_paid               16980367 non-null  int64         
 4   avg_plan_price           16980367 non-null  float64       
 5   total_auto_renew         16980367 non-null  int64         
 6   total_cancel             16980367 non-null  int64         
 7   last_payment_plan_days   16980367 non-null  int64         
 8   last_plan_list_price     16980367 non-null  int64         
 9   last_actual_amount_paid  16980367 non-null  int64         
 10  last_is_auto_renew       16980367 non-null  int64         
 11  last_transaction_date    16980367 non-null  str           


In [3]:
users = trans[['msno', 'cohort_cutoff_date']].copy()

In [4]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 16980367 entries, 0 to 16980366
Data columns (total 2 columns):
 #   Column              Dtype         
---  ------              -----         
 0   msno                str           
 1   cohort_cutoff_date  datetime64[us]
dtypes: datetime64[us](1), str(1)
memory usage: 971.6 MB


In [4]:
del trans

In [5]:
# 1. Initialize Lazy Scan of user logs
lf = pl.scan_csv(
    os.path.join(DATA_DIR, "user_logs.csv"),
    schema_overrides={"date": pl.String} 
).with_columns(
    pl.col("date").str.to_date(format="%Y%m%d")
)

features = ['num_25', 'num_75', 'num_985', 'num_100', 'num_unq', 'total_secs']

def process_cohort_user_log(month, year, users_lazy):
    history_cut_off = datetime(year, month, 1).date()
    start_14 = history_cut_off - timedelta(days=14)
    start_30 = history_cut_off - timedelta(days=30)
    
    
    # 1. Filter raw logs using an Inner Join (much faster than .is_in(list))
    lf_cohort = users_lazy.join(
        lf.filter(pl.col("date") < history_cut_off),
        on="msno",
        how="left"
    )
    
    # Build a single list of expressions to process all metrics simultaneously
    exprs = [
        ((pl.lit(history_cut_off) - pl.col("date").max()).dt.total_days() - 1).alias("days_since_last_use")
    ]
    
    for f in features:
        sum_30 = pl.col(f).filter(pl.col("date") >= start_30).fill_null(0).sum()
        sum_14 = pl.col(f).filter(pl.col("date") >= start_14).fill_null(0).sum()
        
        velocity_expr = (
            pl.when(sum_30 > 0)
            .then(sum_14 / sum_30)
            .otherwise(pl.lit(None))
            .alias(f"{f}_velocity")
        )
        exprs.append(velocity_expr)
    
    # Aggregate everything in ONE pass. No extra branch joins needed.
    metrics_lf = lf_cohort.group_by(["msno", "cohort_cutoff_date"]).agg(exprs)
    
    return metrics_lf

In [6]:
cohort_plans = []
for year in range(2015, 2018):
    for month in range(1, 13):
        # Skip condition
        if year == 2015 and month == 1:
            continue
        
        # Break condition
        if year == 2017 and month == 3:
            break
        
        pandas_cohort_slice = users[users['cohort_cutoff_date'] == datetime(year, month, 1)]
        if pandas_cohort_slice.empty:
            print("no user churn for", month, year)
            continue
            
        current_cohort_users = pl.from_pandas(pandas_cohort_slice).with_columns(
            pl.col("cohort_cutoff_date").cast(pl.Date)
        ).lazy()

        cohort_plans.append(process_cohort_user_log(month, year, current_cohort_users))

In [7]:
del users


In [ ]:
file_num = 0
for plan in cohort_plans:
    file_num += 1
    print("working on", file_num)
    output_file = os.path.join(DATA_DIR, f"mult_cohort_usage_data_{file_num}.parquet")
    plan.sink_parquet(output_file, engine="streaming")
    

working on 25


In [5]:
df = pd.DataFrame()

for file_num in range(1,26):
    output_file = os.path.join(DATA_DIR, f"mult_cohort_usage_data_{file_num}.parquet")
    temp = pd.read_parquet(output_file)
    df = pd.concat([df, temp])


In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 16980367 entries, 0 to 875335
Data columns (total 9 columns):
 #   Column               Dtype  
---  ------               -----  
 0   msno                 str    
 1   cohort_cutoff_date   object 
 2   days_since_last_use  float64
 3   num_25_velocity      float64
 4   num_75_velocity      float64
 5   num_985_velocity     float64
 6   num_100_velocity     float64
 7   num_unq_velocity     float64
 8   total_secs_velocity  float64
dtypes: float64(7), object(1), str(1)
memory usage: 2.0+ GB


In [8]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 16980367 entries, 0 to 875335
Data columns (total 9 columns):
 #   Column               Non-Null Count     Dtype  
---  ------               --------------     -----  
 0   msno                 16980367 non-null  str    
 1   cohort_cutoff_date   16980367 non-null  object 
 2   days_since_last_use  14403986 non-null  float64
 3   num_25_velocity      12964219 non-null  float64
 4   num_75_velocity      12139370 non-null  float64
 5   num_985_velocity     12113051 non-null  float64
 6   num_100_velocity     13083010 non-null  float64
 7   num_unq_velocity     13259308 non-null  float64
 8   total_secs_velocity  13235233 non-null  float64
dtypes: float64(7), object(1), str(1)
memory usage: 2.0+ GB


In [9]:
df.head(100)

,msno,cohort_cutoff_date,days_since_last_use,num_25_velocity,num_75_velocity,num_985_velocity,num_100_velocity,num_unq_velocity,total_secs_velocity
0,GL0NCk+xsxjX69d9sDe4AajTs0Vn51iiViFCZTiNbDw=,2015-02-01,2.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,siClR1LyAUqVVH9bLKFt6F+FWjdqsaz8G8KI/drSSHE=,2015-02-01,0.0,0.607143,0.150000,0.428571,0.389956,0.522843,0.401507
2,BGrqaY1yM37j8ydHwU44l2SZqBh74Hy/ifqkTUhUOPI=,2015-02-01,0.0,0.393939,NaN,0.750000,0.425373,0.452830,0.430844
3,rk0lM3Zw7i61lrqLpX4hegBwH2GTsmn0j9MwdgTYrkM=,2015-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,uRyp9+40qlzADBaSNdvF+jfZDiggmRefpSmWAxEfi1A=,2015-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
95,lk6S2BOC8+6ZNOt+aWLXv5MHbocE/8LW6GHkOLtPL/A=,2015-02-01,1.0,0.204082,0.210526,0.073171,0.168658,0.164292,0.176051
96,Iq5g0vnhIW71wTujUQJBWfuDFeV+im5HnLOkbiSkMZE=,2015-02-01,0.0,0.358491,0.551724,0.454545,0.388489,0.402597,0.402450
97,6ovIiDRkO3QTb8SyNFjT+INsNMNkp6fWHLc2smJEOp8=,2015-02-01,1.0,0.600000,0.333333,0.315789,0.426573,0.405660,0.407855
98,8XFk3QN2kvPCRTZbPLQ2EvjY1bLVbEd7sDnluW4/U0c=,2015-02-01,0.0,0.652482,0.470588,0.636364,0.443820,0.498452,0.456941
